# k-Nearest Neighbors (k-NN) Model

Train and evaluate a k-Nearest Neighbors classifier, find the optimal value of k, and evaluate performance.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
# If features are missing or outdated, automatically re-extract with improved feature set
import os
import sys
import shutil
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("./processed_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

# ---- Version check: delete stale feature cache ----
# Deep features have exactly 512 dims (ResNet18 embeddings)
if npz_file.exists():
    _check = np.load(npz_file)
    if _check['X_train'].shape[1] != 512:
        print(f"[Version check] Outdated feature cache detected ({_check['X_train'].shape[1]} dims). Deleting and extracting 512-dim ResNet18 deep visual features...")
        for _f in DATA_DIR.glob('*.npz'): _f.unlink()
        for _f in DATA_DIR.glob('*.npy'): _f.unlink()
    del _check

if not npz_file.exists():
    print("Feature files not found. Running automatic feature extraction pipeline with cropping...")
    import pandas as pd
    import kagglehub
    from tqdm.auto import tqdm
    from PIL import Image
    from skimage.feature import hog, local_binary_pattern
    import cv2
    
    # ---- Step 1: Download and prepare dataset ----
    def locate_metadata():
        for p in [Path("metadata_preprocessed.csv"), DATA_DIR / "metadata_preprocessed.csv"]:
            if p.exists():
                return p
        return None

    metadata_file = locate_metadata()
    dataset_preprocessed_dir = Path("dataset_20_species_preprocessed")

    if metadata_file is None or not dataset_preprocessed_dir.exists():
        print("  Downloading CUB-200-2011 dataset via Kaggle Hub...")
        download_path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
        DATASET_ROOT = download_path / "CUB_200_2011"
        IMAGES_FOLDER = DATASET_ROOT / "images"
        
        SUBSET_DIR = Path("./dataset_20_species")
        if SUBSET_DIR.exists():
            shutil.rmtree(SUBSET_DIR)
        SUBSET_DIR.mkdir(parents=True, exist_ok=True)
        
        bd_keywords = [
            'Crow', 'Kingfisher', 'Hummingbird', 'Mallard', 'Warbler',
            'Towhee', 'Jay', 'Creeper', 'Waxwing', 'Cuckoo',
            'Thrush', 'Woodpecker', 'Wren', 'Vireo', 'Catbird',
            'Meadowlark', 'Blackbird', 'Gull', 'Tern', 'Pelican'
        ]
        species_folders = sorted([f for f in IMAGES_FOLDER.iterdir() if f.is_dir()])
        selected_species = []
        for folder in species_folders:
            if any(kw.lower() in folder.name.lower() for kw in bd_keywords):
                if folder not in selected_species:
                    selected_species.append(folder)
            if len(selected_species) == 20:
                break
        
        for species_path in selected_species:
            shutil.copytree(str(species_path), SUBSET_DIR / species_path.name)
        
        classes_df = pd.read_csv(DATASET_ROOT / "classes.txt", sep=r"\s+", names=["class_id", "class_name"])
        images_df = pd.read_csv(DATASET_ROOT / "images.txt", sep=r"\s+", names=["image_id", "image_path"])
        labels_df = pd.read_csv(DATASET_ROOT / "image_class_labels.txt", sep=r"\s+", names=["image_id", "class_id"])
        bboxes_df = pd.read_csv(DATASET_ROOT / "bounding_boxes.txt", sep=r"\s+", names=["image_id", "x", "y", "width", "height"])
        
        selected_names = [p.name for p in selected_species]
        dataset_info = images_df.merge(labels_df, on="image_id").merge(classes_df, on="class_id").merge(bboxes_df, on="image_id")
        dataset_info = dataset_info[dataset_info["class_name"].isin(selected_names)].copy().reset_index(drop=True)
        dataset_info["full_image_path"] = dataset_info["image_path"].apply(lambda p: SUBSET_DIR / p)
        
        sys.path.append(str(Path(".").resolve()))
        import importlib
        import src.preprocessing
        importlib.reload(src.preprocessing)
        from src.preprocessing import create_stratified_splits, preprocess_and_save_image
        
        split_metadata = create_stratified_splits(dataset_info, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42)
        
        if dataset_preprocessed_dir.exists():
            shutil.rmtree(dataset_preprocessed_dir)
        dataset_preprocessed_dir.mkdir(parents=True, exist_ok=True)
        
        preprocessed_paths = []
        for idx, row in split_metadata.iterrows():
            dst_file = dataset_preprocessed_dir / Path(row["image_path"])
            bbox = (row["x"], row["y"], row["width"], row["height"])
            preprocess_and_save_image(row["full_image_path"], dst_file, target_size=(224, 224), bbox=bbox)
            preprocessed_paths.append(str(dst_file))
        split_metadata["preprocessed_image_path"] = preprocessed_paths
        cols = ["image_id", "image_path", "class_id", "class_name", "split", "preprocessed_image_path", "x", "y", "width", "height"]
        split_metadata[cols].to_csv("metadata_preprocessed.csv", index=False)
        split_metadata[cols].to_csv(DATA_DIR / "metadata_preprocessed.csv", index=False)
        metadata_file = Path("metadata_preprocessed.csv")
        print("  Dataset preparation complete!")

    # ---- Step 2: Deep Feature Extraction (Pre-trained ResNet18) + Classical Features ----
    import torch
    import torchvision.models as models_tv
    import torchvision.transforms as transforms_tv
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Loading pre-trained ResNet18 feature extractor on {device}...")
    weights = models_tv.ResNet18_Weights.DEFAULT
    resnet = models_tv.resnet18(weights=weights)
    resnet.fc = torch.nn.Identity()  # Remove final FC layer -> 512-dim embedding
    resnet.eval()
    resnet.to(device)
    
    transform_deep = transforms_tv.Compose([
        transforms_tv.Resize((224, 224)),
        transforms_tv.ToTensor(),
        transforms_tv.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def _extract_deep_features(img):
        tensor = transform_deep(img.convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = resnet(tensor).squeeze(0).cpu().numpy()
        return feat.astype(np.float32)

    metadata_df = pd.read_csv(locate_metadata())
    X_deep, y_all, splits_all = [], [], []

    print("  Extracting deep visual embeddings using frozen ResNet18...")
    for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        img_path = Path(row["preprocessed_image_path"])
        img = Image.open(img_path).convert("RGB")
        X_deep.append(_extract_deep_features(img))
        y_all.append(row["class_id"] - 1)
        splits_all.append(row["split"])

    X_deep = np.array(X_deep, dtype=np.float32)
    y_all = np.array(y_all, dtype=np.int32)
    splits_all = np.array(splits_all)

    unique_classes = sorted(metadata_df["class_name"].unique())
    lm = {cls: idx for idx, cls in enumerate(unique_classes)}
    joblib.dump(lm, DATA_DIR / "label_mapping.pkl")

    def _save_split_npz(fname, X):
        np.savez_compressed(
            DATA_DIR / fname,
            X_train=X[splits_all=="train"], y_train=y_all[splits_all=="train"],
            X_val=X[splits_all=="val"],   y_val=y_all[splits_all=="val"],
            X_test=X[splits_all=="test"],  y_test=y_all[splits_all=="test"]
        )
    _save_split_npz("deep_features.npz", X_deep)
    _save_split_npz("combined_hog_color_lbp.npz", X_deep)
    print(f"  Deep feature extraction complete! Features dimension: {X_deep.shape[1]}")

# ---- Load the features ----
data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape} (Standardized)")
print(f"  Validation set: {X_val.shape} (Standardized)")
print(f"  Testing set   : {X_test.shape} (Standardized)")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

## 1. Train Baseline k-NN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("--- Training Baseline k-NN (k=5) ---")
knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train, y_train)

y_pred_base = knn_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_base)
print(f"Baseline k-NN (k=5) Test Accuracy: {baseline_acc * 100:.2f}%")

## 2. Find the Best Value of k & Hyperparameter Tuning

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

print("--- Finding Best Value of k with GridSearchCV ---")
param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search_knn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_knn.fit(X_train, y_train)

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters : {grid_search_knn.best_params_}")
print(f"Best Cross-Val Score : {grid_search_knn.best_score_ * 100:.2f}%")

## 3. Visualize Accuracy vs. Value of k

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
scores_uniform = []
scores_distance = []

results = grid_search_knn.cv_results_
for k in k_values:
    for params, mean_score in zip(results['params'], results['mean_test_score']):
        if params['n_neighbors'] == k and params['metric'] == 'euclidean':
            if params['weights'] == 'uniform':
                scores_uniform.append(mean_score * 100)
            elif params['weights'] == 'distance':
                scores_distance.append(mean_score * 100)

plt.figure(figsize=(10, 5))
plt.plot(k_values, scores_uniform[:len(k_values)], marker='o', label='Uniform Weights (Euclidean)')
plt.plot(k_values, scores_distance[:len(k_values)], marker='s', label='Distance Weights (Euclidean)')
plt.title('k-NN Performance: Cross-Validation Accuracy vs k')
plt.xlabel('Value of k (n_neighbors)')
plt.ylabel('CV Accuracy (%)')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Evaluate & Save Best k-NN Model

In [ ]:
import seaborn as sns

best_knn = grid_search_knn.best_estimator_
y_pred = best_knn.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
print(f"Final Tuned k-NN Test Accuracy: {final_acc * 100:.2f}%")

# Save model
MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "knn_model.pkl"
joblib.dump(best_knn, model_path)
print(f"Best k-NN model saved to '{model_path}' successfully!")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Tuned k-NN')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()